# 01 — Getting started con onpe-eg2026

Este notebook introduce el dataset: estructura, carga básica con Polars, primeras queries.

**Estructura del dataset**:

- `actas_cabecera`: 463,830 filas (92,766 mesas × 5 elecciones). Una fila por acta con totales oficiales.
- `actas_votos`: ~18.6M filas. Una fila por (acta × partido). `nvotos` es el campo principal.
- `actas_votos_tidy`: versión long-format con contexto geográfico ya joinado (recomendado para análisis).
- `actas_candidatos`: array completo de candidatos por lista (clave para Senado/Diputados).
- `actas_linea_tiempo`: eventos de transición de estado por acta.
- `actas_archivos`: metadata de 811k PDFs escaneados.
- `dim_*`: catálogos (mesas, padrón RENIEC, resoluciones, etc.).

**Convención clave**: los ubigeos ONPE coinciden 100% con los UBIGEO_RENIEC (join directo distrito↔distrito).

In [ ]:
import polars as pl

cab = pl.read_parquet('parquet/actas_cabecera.parquet')
print(f'cabecera: {cab.shape[0]:,} filas × {cab.shape[1]} columnas')
cab.head(3)

In [ ]:
# Elecciones capturadas
cab.group_by('idEleccion').agg(pl.len().alias('actas')).sort('idEleccion')

In [ ]:
# Estados del conteo (cuántas actas son terminales vs volátiles)
cab.group_by('codigoEstadoActa').agg(
    pl.len().alias('n'),
    (pl.len() * 100.0 / cab.shape[0]).alias('pct')
).sort('pct', descending=True)

In [ ]:
# Universo exterior (DE 27) vs Perú (DE 1-26)
cab.filter(pl.col('idEleccion') == 10).group_by('idAmbitoGeografico').agg(
    pl.len().alias('actas'),
    pl.col('totalElectoresHabiles').sum().alias('electores')
)

## Próximos pasos

- `02-participacion-por-region.ipynb`: análisis de participación ciudadana
- `03-resultados-presidencial.ipynb`: resultados nacionales + top partidos
- `04-voto-exterior.ipynb`: análisis del DE 27 (peruanos en el extranjero)
- `05-anomaly-detection.ipynb`: detector de anomalías estadísticas